# Core-only Tutorial: Synthetic Optimisation with alf-core

This tutorial runs a complete active learning loop using **only `alf-core`** — no PyTorch required.

We'll optimise a synthetic function over a discrete sequence space using:
- a custom `BaseDataset` subclass backed by a numpy array
- a simple Gaussian Process surrogate built with scipy
- a greedy acquisition function
- ALF's standard `DesignTask` to drive the loop

This is a good starting point for understanding ALF's abstractions before adding the
complexity of PyTorch models.

**Requirements:** `alf-core`, `numpy`, `scipy` (no PyTorch needed)

## Installation

```bash
pip install git+https://github.com/instadeepai/alf.git#subdirectory=core
```

In [ ]:
import numpy as np
from alf_core import (
    AcquisitionFunction,
    BaseModel,
    Candidate,
    DatasetSearch,
    DesignTask,
    LabelledCandidates,
    Modality,
    Optimizer,
    Oracle,
    Predictions,
    Surrogate,
    TerminalStateLogger,
)
from alf_core.dataset.base_dataset import BaseDataset, BaseDatasetConfig
from alf_core.utils.enums import ProblemType
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel

## 1. Define a synthetic dataset

Our search space is a grid of 2-D points. The true objective is a noisy Branin function —
a common benchmark with two near-optimal regions. We treat each point as a `Candidate` with
a 2-element numpy array as its value.

`BaseDataset` expects us to implement `load_dataset` (returns the full pool) and `query`
(scores a batch of candidates — this is what the oracle calls each round).

In [ ]:
def branin(x: np.ndarray) -> float:
    """Branin function, scaled so the global optimum is near 1.0."""
    x1, x2 = x[0] * 15 - 5, x[1] * 15  # rescale [0,1]^2 to Branin domain
    a, b, c = 1.0, 5.1 / (4 * np.pi**2), 5 / np.pi
    r, s, t = 6.0, 10.0, 1 / (8 * np.pi)
    raw = a * (x2 - b * x1**2 + c * x1 - r) ** 2 + s * (1 - t) * np.cos(x1) + s
    return float(1.0 - raw / 300)  # invert and normalise so higher is better


class SyntheticDatasetConfig(BaseDatasetConfig):
    grid_size: int = 50


class SyntheticDataset(BaseDataset):
    """A discrete grid over [0,1]^2 scored by the Branin function."""

    config: SyntheticDatasetConfig

    def load_dataset(self) -> LabelledCandidates:
        rng = np.random.default_rng(self.config.seed)
        n = self.config.grid_size**2
        xs = np.array([
            (i / self.config.grid_size, j / self.config.grid_size)
            for i in range(self.config.grid_size)
            for j in range(self.config.grid_size)
        ])
        # Add a tiny shuffle so train/test splits are not spatially biased
        idx = rng.permutation(n)
        xs = xs[idx]
        candidates = [Candidate(id=str(i), value=x) for i, x in enumerate(xs)]
        labels = np.array([branin(x) for x in xs])
        return LabelledCandidates(candidates=candidates, labels=labels)

    def query(self, candidates: list[Candidate]) -> np.ndarray:
        return np.array([branin(c.value) for c in candidates])

## 2. Define a GP model

`BaseModel` requires three methods: `featurise` (convert `Candidate` objects to a feature
matrix), `train` (fit the model on labelled data), and `predict` (return mean and std).

Here we use scikit-learn's `GaussianProcessRegressor`, which only depends on numpy/scipy.

In [ ]:
class GPModel(BaseModel):
    problem_type = ProblemType.REGRESSION
    output_dim = 1

    def __init__(self) -> None:
        kernel = RBF(length_scale=0.1) + WhiteKernel(noise_level=0.01)
        self._gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=3)

    def featurise(self, inputs: list[Candidate]) -> np.ndarray:
        return np.array([c.value for c in inputs])

    def train(
        self,
        train_data: LabelledCandidates,
        val_data: LabelledCandidates | None = None,
        train_config=None,
    ):
        X = self.featurise(train_data.candidates)
        self._gp.fit(X, train_data.labels)

    def predict(self, inputs: list[Candidate]) -> Predictions:
        X = self.featurise(inputs)
        mean, std = self._gp.predict(X, return_std=True)
        return Predictions(mean=mean, std=std)

    def sample(self, inputs: list[Candidate], n_samples: int = 1) -> np.ndarray:
        X = self.featurise(inputs)
        return self._gp.sample_y(X, n_samples=n_samples)

## 3. Define an acquisition function

We use **Upper Confidence Bound (UCB)**: score each candidate as `mean + beta * std`,
balancing exploitation (high mean) with exploration (high uncertainty).

In [ ]:
class UCBAcquisition(AcquisitionFunction):
    def __init__(self, beta: float = 2.0) -> None:
        self.beta = beta

    def __call__(self, predictions: Predictions) -> np.ndarray:
        return predictions.mean + self.beta * predictions.std

## 4. Assemble and run the active learning loop

With all components defined, we wire them together using ALF's standard `DesignTask`.
The task handles the loop: train surrogate → score candidates → acquire batch → query oracle → repeat.

In [ ]:
config = SyntheticDatasetConfig(
    name="branin",
    modality=Modality.TABULAR,
    seed=42,
    train_ratio=0.05,  # start with 5% of the grid as labelled data
    validation_frac=0.0,
    test_ratio=0.1,
    problem_type=ProblemType.REGRESSION,
    grid_size=20,  # 400 candidates total (keep small for demo speed)
)

dataset = SyntheticDataset(config=config)
surrogate = Surrogate(model=GPModel())
optimizer = Optimizer(acquisition_fn=UCBAcquisition(beta=2.0), search_fn=DatasetSearch())
oracle = Oracle(scorer=dataset)

task = DesignTask(num_acq_rounds=5, acq_batch_size=10)
state = task.setup(dataset=dataset, surrogate=surrogate)
task.run(
    state=state,
    state_loggers=[TerminalStateLogger()],
    optimizer=optimizer,
    oracle=oracle,
)

## 5. Inspect results

After the loop completes, `state` holds the full history. We can inspect the best candidate
found across all rounds and see how the maximum improved over time.

In [ ]:
# Best candidate found across all rounds
all_labelled = state.labelled_candidates
best_idx = np.argmax(all_labelled.labels)
print(f"Best score found: {all_labelled.labels[best_idx]:.4f}")
print(f"At point: {all_labelled.candidates[best_idx].value}")
print(f"Global optimum (Branin): ~{branin(np.array([0.54, 0.15])):.4f}")

## Next steps

- Swap `UCBAcquisition` for Thompson Sampling or Expected Improvement
- Replace `GPModel` with a neural network from `alf-tools`
- Try a real dataset: see the [Offline Design Tutorial](https://github.com/instadeepai/alf/blob/main/tutorials/experiments/offline_design_tutorial.ipynb)
- Add your own model or dataset: see the [How-to / Recipes](https://instadeepai.github.io/alf/how-to/index.html)